In [3]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os

## Consolidación de atributos de OSM

Este notebook reúne los resultados de los pasos anteriores para construir una sola versión de la red OSM con atributos de capacidad y velocidad.

Hasta este punto, la información quedó repartida en varias capas:
- `osm_links` contiene los links que ya recibieron atributos por expansión a lo largo de las avenidas;
- `osm_cuchillas_avenidas` contiene los conectores o cuchillas que heredaron atributos de TransCAD;
- `osm_laterales_avenidas` contiene los laterales que también recibieron atributos de TransCAD.

La idea de esta etapa es unir toda esa información en una sola red, creando columnas finales de:
- `cap_final`
- `velprom_final`
- `limvel_final`

Para cada link se conserva primero el atributo disponible en alguna de las capas anteriores.  
Si todavía quedan valores vacíos, después se rellenan usando la mediana de los atributos según el tipo de vialidad principal.

In [2]:
folder = r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/VisumLinks With TransCAD Atts'

In [6]:
# output de notebook 03_propagar_cap_vel_a_lo_largo
osm_links = gpd.read_file(os.path.join(folder, "OSM Links con velocidad y capacidades (a lo largo)", "osm_links_con_atts_a_lo_largo.shp"))

# outputs de notebook 04_laterales_y_cuchillas_fromTransCAD
osm_cuchillas_avenidas = gpd.read_file(os.path.join(folder, "OSM Cuchillas con atributos", "osm_cuchillas_buffers_avenidas_principales_transcad.shp"))
osm_laterales_avenidas = gpd.read_file(os.path.join(folder, "OSM Laterales con atributos", "osm_laterales_buffers_avenidas_principales_transcad.shp"))

In [8]:
osm_links = osm_links.merge(
    osm_cuchillas_avenidas[['u', 'v', 'key', 'conn_cap', 'conn_velpr', 'conn_limve']],
    on= ['u', 'v', 'key'],
    how='left'
)

In [12]:
osm_links = osm_links.merge(
    osm_laterales_avenidas[['u', 'v', 'key', 'lat_cap', 'lat_velpro', 'lat_limvel']],
    on= ['u', 'v', 'key'],
    how='left'
)

In [13]:
osm_links

,u,v,key,osmid,highway,lanes,name,oneway,ref,reversed,...,cap_final,velprom_fi,limvel_fin,geometry,conn_cap,conn_velpr,conn_limve,lat_cap,lat_velpro,lat_limvel
0,267537966,7306651630,0,"[832652768, 619339782, 688424559, 188974544, 1...",motorway,"['3', '2']",Autopista Guadalajara - Morelia,True,MEX 15D;MEX 80D,False,...,3250.0,34.000061,80.0,"LINESTRING (-103.24632 20.61606, -103.24734 20...",NaN,NaN,NaN,NaN,NaN,NaN
1,267537966,5837556433,0,694566994,motorway_link,1,NaN,True,NaN,False,...,NaN,NaN,NaN,"LINESTRING (-103.24632 20.61606, -103.24648 20...",1500.0,15.999973,30.0,NaN,NaN,NaN
2,267538751,273140976,0,189118222,motorway,2,Autopista Guadalajara - Zapotlanejo,True,MEX 80D;MEX 90D,False,...,5000.0,53.999989,80.0,"LINESTRING (-103.1364 20.60565, -103.13432 20....",NaN,NaN,NaN,NaN,NaN,NaN
3,267538751,1746293763,0,907206852,motorway_link,1,Autopista Guadalajara - Morelia,True,MEX 15D,False,...,NaN,NaN,NaN,"LINESTRING (-103.1364 20.60565, -103.1354 20.6...",1500.0,15.999973,30.0,NaN,NaN,NaN
4,273140976,1997658287,0,835083267,motorway,3,Autopista Guadalajara - Zapotlanejo,True,MEX 80D;MEX 90D,False,...,5000.0,53.999989,80.0,"LINESTRING (-103.13185 20.60758, -103.13162 20...",NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495561,13656279161,5538433659,0,577828524,NaN,NaN,Linea 3 del Tren Eléctrico Urbano,False,NaN,False,...,NaN,NaN,NaN,"LINESTRING (-103.39072 20.7291, -103.39086 20....",NaN,NaN,NaN,NaN,NaN,NaN
495562,13656279161,5538433658,0,577828524,NaN,NaN,Linea 3 del Tren Eléctrico Urbano,False,NaN,True,...,NaN,NaN,NaN,"LINESTRING (-103.39072 20.7291, -103.39055 20....",NaN,NaN,NaN,NaN,NaN,NaN
495563,13705424642,4591349866,0,463965070,NaN,NaN,Línea 2 del Tren Eléctrico Urbano,False,NaN,False,...,NaN,NaN,NaN,"LINESTRING (-103.3559 20.6749, -103.35637 20.6...",NaN,NaN,NaN,NaN,NaN,NaN
495564,13705424642,4591553254,0,463965070,NaN,NaN,Línea 2 del Tren Eléctrico Urbano,False,NaN,True,...,NaN,NaN,NaN,"LINESTRING (-103.3559 20.6749, -103.35588 20.6...",NaN,NaN,NaN,NaN,NaN,NaN


- tener solo 3 columnas (cap_final, velprom_final, limvel_final)
- donde conn_cap, conn_velprom, conn_limvel tenga valor pasarlas a la columnas (cap_final, velprom_final, limvel_final) que estan NaN
- donde lat_cap, lat_velprom, lat_limvel tenga valor pasarlas a la columnas (cap_final, velprom_final, limvel_final) que estan NaN

- despues de eso ver cuales aun no tienen cap_final, velprom_final, limvel_final y ver los highway.value_counts() de esas que no tienen valor en esas 3 columnas
- deberia de haber varias de tipos primary, secondary, etc (los principales)
- rellenar esas 3 columnas cap_final, velprom_final, limvel_final para aquellos con highways principales con la mediana de esas 3 columnas para la misma highway

In [ ]:
highways_vias_principales = [
    "primary",
    "secondary",
    "tertiary",
    "trunk",
    "unclassified",
    "motorway",
]

- ej. hubo 80 links con highway=primary que tienen nan en cap_final, velprom_final, limvel_final, sacar la mediana de cap_final, velprom_final, limvel_final para highway=primary y llenar con esa mediana los 80 links

- hacer lo mismo para los de 
- ej. 40 primary_links aun no tienen cap_final, velprom_final, limvel_final, sacar la mediana de cap_final, velprom_final, limvel_final para highway='primary_link' y llenar con esa los 40 links

In [ ]:
highways_cuchilla = ["primary_link", "trunk_link", "motorway_link", "secondary_link", "tertiary_link"]